# Plot u- and g-band Light Curves vs MJD — Comparison with Calibration Parameters and AuxTel Aerosol

This notebook is a variant of `10d_PlotLCwithMJDComparewithCalibs_auxtel_bandri.ipynb`
**applied to the u and g bands instead of r and i**, and comparing the
FGCM aerosol optical depth column (`fgcm_tau`) against the AuxTel/Spectractor
**aerosol** atmospheric parameter (`VAOD`, Vertical Aerosol Optical Depth),
instead of ozone.

For each star, a single **tall, one-column figure** (format: **6 rows x 1
column**) is produced, with the following panels sharing the same MJD
x-axis:

1. `psfFlux` (**u band**) — median +/- sigma_IQR envelope, as in notebook 03/10.
2. `psfFlux` (**g band**) — median +/- sigma_IQR envelope, as in notebook 03/10.
3. **Aerosol comparison** — FGCM aerosol optical depth (`fgcm_tau`, per visit
   of this star, for both the u-band and g-band visits) *and* AuxTel aerosol
   (`VAOD_ram` column of the AuxTel atmospheric table), restricted to the
   AuxTel `FILTER` values **OG550** and **empty**, shown in different
   colours/markers.
4. `calib_local` — u-band and g-band visits overlaid (different colours).
5. `zeropoint` — u-band and g-band visits overlaid (different colours).
6. `airmass` — u-band and g-band visits overlaid (different colours).

**Note on MJD in the AuxTel file:** the AuxTel atmospheric table *does*
contain an `MJD` column (one row per AuxTel exposure) — it is used directly,
no timestamp conversion needed.

**Note on the AuxTel `FILTER` column:** the raw values are not exactly
`"OG550"` / `"empty"` (e.g. `"OG550_65mm_1"`), so a small helper function
classifies each row into `"OG550"`, `"empty"`, or `None` (discarded).

**Note on the AuxTel aerosol column:** as for ozone, the AuxTel table
provides two independent fit results for the aerosol optical depth,
`VAOD_ram` and `VAOD_rum`. This notebook uses the **`_ram`** fit
(`VAOD_ram` / `VAOD_err_ram`), for consistency with notebook 10d.

**Outlier highlighting:** as in notebook 10/10b/10c/10d, `psfFlux` points
deviating from the median by more than `OUTLIER_NSIGMA * sigma_IQR` are
flagged **independently for each band** (u and g have their own
median/sigma_IQR). Outliers are marked with a **vertical dashed line** at the
outlier MJD(s) across *all* panels (union of u-band and g-band outlier
epochs), plus a red star marker on the panels built from per-visit rows
(psfFlux u, psfFlux g, FGCM aerosol, calib_local, zeropoint, airmass)
whenever that row's MJD matches an outlier epoch of its own band.

---
- **Author:** Sylvie Dagoret-Campagne
- **Affiliation:** IJCLab/IN2P3/CNRS, Université Paris-Saclay
- **Created:** 2026-07-06
- **Last update:** 2026-07-06
- **Reference notebook:** `10d_PlotLCwithMJDComparewithCalibs_auxtel_bandri.ipynb`


## 1. Imports

In [ ]:
import gc
import logging
import os
import sys

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import AutoMinorLocator

from astropy.time import Time

In [ ]:
# Show all rows/columns when displaying pandas tables
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

In [ ]:
# Enable interactive matplotlib backend if ipympl is available
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found -> interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found -> %matplotlib inline")

In [ ]:
matplotlib.rcParams["figure.max_open_warning"] = 50

## 2. Logging

In [ ]:
log = logging.getLogger()
log.setLevel(logging.INFO)
if not log.handlers:
    handler = logging.StreamHandler(sys.stdout)
    handler.setLevel(logging.INFO)
    formatter = logging.Formatter("%(asctime)s - %(levelname)s - %(message)s")
    handler.setFormatter(formatter)
    log.addHandler(handler)
log.info("Logging initialised.")

## 3. Configuration

In [ ]:
# ── Notebook tag ───────────────────────────────────────────────────
NB_TAG = "PlotLCCompareCalibs_AuxtelBandUG_10e"

# ── Input: FGCM-enriched per-star light curves (output of notebook 08) ──
DIR_DATA_IN = "./data_AddFGCM_08_out"
DIR_PER_STAR_IN = os.path.join(DIR_DATA_IN, "per_star")
PER_STAR_SUFFIX = "_lc_fgcm.csv"

# ── Input: AuxTel atmospheric (Spectractor) table ─────────────────────────
DIR_AUXTEL = "./data_auxtelspectro"
AUXTEL_FILE = "keep_auxtel_atmosphere_feb26_gaiaspec_gaiatarget_calspecthroughput_filteredtightcuts.csv"
AUXTEL_MJD_COL = "MJD"
AUXTEL_AEROSOL_COL = "VAOD_ram"
AUXTEL_AEROSOL_ERR_COL = "VAOD_err_ram"
AUXTEL_FILTER_COL = "FILTER"

# ── Output figures ────────────────────────────────────────────────────
DIR_FIGS = f"./figs_{NB_TAG}"
os.makedirs(DIR_FIGS, exist_ok=True)
log.info("Figure directory: %s", DIR_FIGS)

# ── Photometric / calibration columns ───────────────────────────────────
MJD_COL = "expMidptMJD"
FLUX_COL = "psfFlux"
FLUX_ERR_COL = "psfFluxErr"
BAND_COL = "band"
CALIB_LOCAL_COL = "calib_local"
ZEROPOINT_COL = "zeropoint"
AIRMASS_COL = "airmass"
FGCM_AEROSOL_COL = "fgcm_tau"

# ── This notebook processes BOTH the u and g bands ──────────────────────
BAND_U = "u"
BAND_G = "g"
BAND_COLORS = {
    BAND_U: "#6C2DC7",  # purple, same convention as notebook 04 for 'u'
    BAND_G: "#1CB94A",  # green, same convention as notebook 04 for 'g'
}
BAND_MARKERS = {
    BAND_U: "o",
    BAND_G: "s",
}

# ── Colours for the calibration / atmospheric panels (per-quantity, shared
#    across bands — the actual band colour is used for the data points) ───
ROW_COLORS = {
    "fgcm_tau": "grey",
    "calib_local": "purple",
    "zeropoint": "royalblue",
    "airmass": "dimgray",
}

# ── AuxTel filter groups to keep, and their plotting colours/markers ──────
# Raw FILTER values are not exact ("OG550_65mm_1", etc.); see
# classify_auxtel_filter() below for the matching logic.
AUXTEL_FILTER_STYLE = {
    "OG550": {"color": "tab:red", "marker": "o", "label": "AuxTel Aerosol (OG550)"},
    "empty": {"color": "tab:blue", "marker": "s", "label": "AuxTel Aerosol (empty)"},
}

# ── Y-axis clipping for the psfFlux panels ────────────────────────────────
# median +/- N_SIGMA_YLIM * sigma_IQR   (sigma_IQR = IQR / 1.349)
N_SIGMA_YLIM = 5.0

# ── Outlier threshold: points beyond this many sigma_IQR from the median
# are flagged with a vertical dashed line across all panels.
OUTLIER_NSIGMA = 3.0

# ── Minimum number of good points (per band) to plot that band's flux panel
MIN_POINTS = 5

# ── Figure size (width, height) in inches — single tall column per star ──
FIG_WIDTH = 12
FIG_HEIGHT = 16

## 4. Helper functions

In [ ]:
# ── Robust sigma from the inter-quartile range ────────────────────────
def sigma_iqr(values):
    """Robust standard-deviation estimate via the inter-quartile range.
    sigma_IQR = IQR / 1.3489795
    """
    q75, q25 = np.nanpercentile(values, [75, 25])
    return (q75 - q25) / 1.3489795


# ── savefig: PDF + PNG ───────────────────────────────────────────────
def savefig(fig, name, dpi=150):
    """Save *fig* as both PDF and PNG under DIR_FIGS."""
    base = os.path.join(DIR_FIGS, name)
    fig.savefig(base + ".pdf", dpi=dpi, bbox_inches="tight")
    fig.savefig(base + ".png", dpi=dpi, bbox_inches="tight")
    log.info("Saved figure: %s (.pdf/.png)", base)
    # plt.close(fig)


# ── MJD <-> matplotlib date number (linear day-count shift) ──────────────
MJD_TO_MPL_OFFSET = mdates.date2num(Time(0.0, format="mjd").to_datetime())


def _mjd_to_mpl(mjd):
    return np.asarray(mjd, dtype=float) + MJD_TO_MPL_OFFSET


def _mpl_to_mjd(mpl):
    return np.asarray(mpl, dtype=float) - MJD_TO_MPL_OFFSET


def add_date_top_axis(ax):
    """Add a secondary top x-axis showing calendar dates (YYYY-MM-DD)."""
    secax = ax.secondary_xaxis("top", functions=(_mjd_to_mpl, _mpl_to_mjd))
    secax.xaxis.set_major_locator(mdates.AutoDateLocator())
    secax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
    plt.setp(secax.get_xticklabels(), rotation=30, ha="left", fontsize=6)
    return secax


# ── Classify AuxTel FILTER strings into "OG550" / "empty" / None ─────────
def classify_auxtel_filter(filter_series: pd.Series) -> pd.Series:
    """Map raw AuxTel FILTER strings (e.g. "OG550_65mm_1", "empty") to a
    clean group label in {"OG550", "empty"}, or None if it matches neither.
    """

    def _classify(v):
        if pd.isna(v):
            return None
        s = str(v).strip().lower()
        if s == "empty":
            return "empty"
        if "og550" in s:
            return "OG550"
        return None

    return filter_series.apply(_classify)


def select_band(df: pd.DataFrame, band: str) -> pd.DataFrame:
    """Select rows for *band*, requiring valid MJD, psfFlux, and (if present)
    a strictly positive psfFluxErr. Returns rows sorted by MJD."""
    mask = (df[BAND_COL] == band) & df[MJD_COL].notna() & df[FLUX_COL].notna()
    if FLUX_ERR_COL in df.columns:
        mask &= df[FLUX_ERR_COL].notna() & (df[FLUX_ERR_COL] > 0)
    return df[mask].sort_values(MJD_COL)


def band_flux_stats(sub: pd.DataFrame):
    """Return (mjd, flux, err, med, sig_iqr, outlier_mask, outlier_mjds) for
    one band's selected rows *sub* (as returned by select_band)."""
    mjd = sub[MJD_COL].values
    flux = sub[FLUX_COL].values
    err = sub[FLUX_ERR_COL].values if FLUX_ERR_COL in sub.columns else np.zeros_like(flux)
    med = np.nanmedian(flux)
    sig_iqr = sigma_iqr(flux)
    if sig_iqr == 0:
        sig_iqr = np.nanstd(flux) or 1.0
    outlier_mask = np.abs(flux - med) > OUTLIER_NSIGMA * sig_iqr
    outlier_mjds = mjd[outlier_mask]
    return mjd, flux, err, med, sig_iqr, outlier_mask, outlier_mjds

## 5. Load the AuxTel atmospheric (aerosol) table

Only the columns needed here (`MJD`, `VAOD_ram`, `VAOD_err_ram`,
`FILTER`) are read from the ~50 MB AuxTel table to keep memory usage low.
Rows are then restricted to the `OG550` / `empty` filter groups.


In [ ]:
auxtel_path = os.path.join(DIR_AUXTEL, AUXTEL_FILE)
log.info("Loading AuxTel atmospheric table: %s", auxtel_path)

df_auxtel_raw = pd.read_csv(
    auxtel_path,
    usecols=[AUXTEL_MJD_COL, AUXTEL_AEROSOL_COL, AUXTEL_AEROSOL_ERR_COL, AUXTEL_FILTER_COL],
)
log.info("AuxTel table loaded: %d rows.", len(df_auxtel_raw))

df_auxtel_raw["filter_group"] = classify_auxtel_filter(df_auxtel_raw[AUXTEL_FILTER_COL])
df_auxtel = df_auxtel_raw.dropna(subset=[AUXTEL_MJD_COL, AUXTEL_AEROSOL_COL, "filter_group"]).copy()
df_auxtel = df_auxtel[df_auxtel["filter_group"].isin(AUXTEL_FILTER_STYLE.keys())]
df_auxtel = df_auxtel.sort_values(AUXTEL_MJD_COL)

log.info(
    "AuxTel aerosol rows kept after filter selection (OG550/empty): %d  |  MJD range: %.2f - %.2f",
    len(df_auxtel),
    df_auxtel[AUXTEL_MJD_COL].min(),
    df_auxtel[AUXTEL_MJD_COL].max(),
)
df_auxtel["filter_group"].value_counts()

## 6. Core plotting function: one star -> one 6x1 column figure

Panels (top to bottom): `psfFlux` (u band), `psfFlux` (g band), aerosol
comparison (FGCM + AuxTel OG550/empty), `calib_local`, `zeropoint`,
`airmass`. The last three panels overlay both u-band and g-band visits
(different colours). All panels share the same MJD x-axis.


In [ ]:
def plot_star_ug_calib_auxtel(
    df_star: pd.DataFrame, simbad_id: str, file_stem: str, df_auxtel: pd.DataFrame
) -> None:
    """Build and save the 6x1 figure comparing the u-band and g-band psfFlux
    light curves with calibration / atmospheric quantities and the
    independent AuxTel aerosol measurements, for one stable star.

    Parameters
    ----------
    df_star    : per-star FGCM-enriched light-curve DataFrame.
    simbad_id  : human-readable identifier shown in the figure title.
    file_stem  : output filename stem (no extension, no directory).
    df_auxtel  : pre-filtered AuxTel aerosol table (see Section 5).
    """
    df = df_star.dropna(subset=[MJD_COL, FLUX_COL]).copy()

    sub_u = select_band(df, BAND_U)
    sub_g = select_band(df, BAND_G)
    n_pts_u = len(sub_u)
    n_pts_g = len(sub_g)

    fig, axes = plt.subplots(6, 1, figsize=(FIG_WIDTH, FIG_HEIGHT), sharex=True)
    ax_flux_u, ax_flux_g, ax_aerosol, ax_calib, ax_zp, ax_am = axes

    ra = df["ra"].mean() if "ra" in df.columns else np.nan
    dec = df["dec"].mean() if "dec" in df.columns else np.nan
    fig.suptitle(
        f"{simbad_id} | (ra,dec)=({ra:.2f},{dec:.2f}) | bands 'u'+'g'   |   "
        f"outliers: |flux - median| > {OUTLIER_NSIGMA:.0f}*sigma_IQR (per band)",
        fontsize=13,
        fontweight="bold",
        y=1.0,
    )

    # ── Empty star (not enough points in EITHER band) ──────────────────────
    if n_pts_u < MIN_POINTS and n_pts_g < MIN_POINTS:
        for ax in axes:
            ax.text(
                0.5,
                0.5,
                f"u: {n_pts_u} pt(s)  |  g: {n_pts_g} pt(s)",
                ha="center",
                va="center",
                transform=ax.transAxes,
                color="grey",
                fontsize=11,
            )
            ax.set_xticks([])
            ax.set_yticks([])
        savefig(fig, file_stem)
        gc.collect()
        return

    # ── Per-band flux statistics (independent median / sigma_IQR / outliers) ─
    stats_u = band_flux_stats(sub_u) if n_pts_u >= MIN_POINTS else None
    stats_g = band_flux_stats(sub_g) if n_pts_g >= MIN_POINTS else None

    # ── Shared MJD range across all 6 panels, from whichever band(s) exist ──
    mjd_all = np.concatenate([s[0] for s in (stats_u, stats_g) if s is not None])
    mjd_pad = max((mjd_all.max() - mjd_all.min()) * 0.03, 0.5)
    xlim_lo, xlim_hi = mjd_all.min() - mjd_pad, mjd_all.max() + mjd_pad

    # ── Union of outlier MJDs across both bands (vertical lines, all panels) ─
    outlier_mjds_all = np.concatenate([s[6] for s in (stats_u, stats_g) if s is not None])
    n_outliers_all = len(outlier_mjds_all)

    # ═══ Panels 1-2: psfFlux (u band, g band) — median +/- sigma_IQR ═══════
    flux_panels = [
        (ax_flux_u, BAND_U, stats_u, n_pts_u),
        (ax_flux_g, BAND_G, stats_g, n_pts_g),
    ]
    for ax, band, stats, n_pts in flux_panels:
        color = BAND_COLORS[band]
        if stats is None:
            ax.text(
                0.5,
                0.5,
                f"{band} band\n{n_pts} point(s)",
                ha="center",
                va="center",
                transform=ax.transAxes,
                color="grey",
                fontsize=11,
            )
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_ylabel(f"{band} psfFlux [nJy]", fontsize=9, color=color)
            continue

        mjd, flux, err, med, sig_iqr, outlier_mask, outlier_mjds = stats
        n_outliers = int(outlier_mask.sum())

        ax.errorbar(
            mjd,
            flux,
            yerr=err,
            fmt="o",
            ms=4,
            color=color,
            ecolor=color,
            alpha=0.8,
            elinewidth=0.8,
            capsize=2,
            zorder=2,
            label=f"N={n_pts}",
        )
        ax.axhline(med, color=color, lw=1.4, ls="--", alpha=0.9, zorder=3)
        ax.axhspan(med - sig_iqr, med + sig_iqr, color=color, alpha=0.12, zorder=1)
        if n_outliers > 0:
            ax.scatter(
                mjd[outlier_mask],
                flux[outlier_mask],
                marker="*",
                s=100,
                facecolor="red",
                edgecolor="k",
                linewidth=0.6,
                zorder=5,
                label=f"outliers ({n_outliers})",
            )
        ylim_lo = med - N_SIGMA_YLIM * sig_iqr
        ylim_hi = med + N_SIGMA_YLIM * sig_iqr
        if ylim_hi - ylim_lo < 1e-10 * abs(med) + 1.0:
            ylim_lo, ylim_hi = med - 1.0, med + 1.0
        ax.set_ylim(ylim_lo, ylim_hi)
        rel_scatter = 100.0 * sig_iqr / abs(med) if abs(med) > 0 else np.nan
        ax.text(
            0.02,
            0.94,
            f"sigma_IQR={sig_iqr:.1f} nJy ({rel_scatter:.2f}%)",
            transform=ax.transAxes,
            fontsize=7,
            va="top",
            ha="left",
            color=color,
        )
        ax.legend(loc="upper right", fontsize=7, framealpha=0.6)
        ax.set_ylabel(f"{band} psfFlux [nJy]", fontsize=9, color=color)
        ax.set_title(f"psfFlux ({band} band, {n_pts} pts)", fontsize=10)

    add_date_top_axis(ax_flux_u)

    # ═══ Panel 3: aerosol comparison — FGCM (u+g visits) + AuxTel OG550/empty ═
    for band, stats in ((BAND_U, stats_u), (BAND_G, stats_g)):
        if stats is None or FGCM_AEROSOL_COL not in (sub_u if band == BAND_U else sub_g).columns:
            continue
        sub_band = sub_u if band == BAND_U else sub_g
        mjd_b = stats[0]
        outlier_mask_b = stats[5]
        aerosol_fgcm = sub_band[FGCM_AEROSOL_COL].values
        valid_fgcm = np.isfinite(aerosol_fgcm)
        if not valid_fgcm.any():
            continue
        ax_aerosol.scatter(
            mjd_b[valid_fgcm],
            aerosol_fgcm[valid_fgcm],
            s=18,
            color=ROW_COLORS["fgcm_tau"] if band == BAND_U else "black",
            marker=BAND_MARKERS[band],
            alpha=0.8,
            zorder=3,
            label=f"FGCM aerosol ({band} visits)",
        )
        outl_fgcm = outlier_mask_b & valid_fgcm
        if outl_fgcm.any():
            ax_aerosol.scatter(
                mjd_b[outl_fgcm],
                aerosol_fgcm[outl_fgcm],
                marker="*",
                s=100,
                facecolor="red",
                edgecolor="k",
                linewidth=0.6,
                zorder=5,
            )

    for group, style in AUXTEL_FILTER_STYLE.items():
        sub_aux = df_auxtel[
            (df_auxtel["filter_group"] == group)
            & (df_auxtel[AUXTEL_MJD_COL] >= xlim_lo)
            & (df_auxtel[AUXTEL_MJD_COL] <= xlim_hi)
        ]
        if len(sub_aux) == 0:
            continue
        yerr_aux = sub_aux[AUXTEL_AEROSOL_ERR_COL] if AUXTEL_AEROSOL_ERR_COL in sub_aux.columns else None
        ax_aerosol.errorbar(
            sub_aux[AUXTEL_MJD_COL],
            sub_aux[AUXTEL_AEROSOL_COL],
            yerr=yerr_aux,
            fmt=style["marker"],
            ms=4,
            color=style["color"],
            ecolor=style["color"],
            alpha=0.75,
            elinewidth=0.6,
            capsize=1.5,
            zorder=2,
            label=f"{style['label']} (N={len(sub_aux)})",
        )

    ax_aerosol.set_ylabel("Aerosol VAOD", fontsize=9)
    ax_aerosol.set_title("FGCM (u+g visits) vs. AuxTel Aerosol (VAOD)", fontsize=10)
    ax_aerosol.legend(loc="best", fontsize=6.5, framealpha=0.6, ncol=2)

    # ═══ Panels 4-6: calib_local / zeropoint / airmass — u+g overlaid ══════
    row_specs = [
        (ax_calib, CALIB_LOCAL_COL, "calib_local"),
        (ax_zp, ZEROPOINT_COL, "zeropoint [mag]"),
        (ax_am, AIRMASS_COL, "airmass"),
    ]
    for ax, col_name, ylabel in row_specs:
        any_data = False
        for band, sub_band, stats in ((BAND_U, sub_u, stats_u), (BAND_G, sub_g, stats_g)):
            if stats is None or col_name not in sub_band.columns:
                continue
            mjd_b = stats[0]
            outlier_mask_b = stats[5]
            vals = sub_band[col_name].values
            valid = np.isfinite(vals)
            if valid.sum() == 0:
                continue
            any_data = True
            color = BAND_COLORS[band]
            ax.scatter(
                mjd_b[valid],
                vals[valid],
                s=14,
                color=color,
                alpha=0.75,
                marker=BAND_MARKERS[band],
                zorder=2,
                label=f"{band} band",
            )
            med_val = np.nanmedian(vals[valid])
            ax.axhline(med_val, color=color, lw=1.1, ls="--", alpha=0.7, zorder=1)
            outl_here = outlier_mask_b & valid
            if outl_here.any():
                ax.scatter(
                    mjd_b[outl_here],
                    vals[outl_here],
                    marker="*",
                    s=100,
                    facecolor="red",
                    edgecolor="k",
                    linewidth=0.6,
                    zorder=5,
                )
        if not any_data:
            ax.text(
                0.5,
                0.5,
                f"'{col_name}' not available",
                ha="center",
                va="center",
                transform=ax.transAxes,
                color="grey",
                fontsize=8,
            )
        else:
            ax.legend(loc="best", fontsize=6.5, framealpha=0.6)
        ax.set_ylabel(ylabel, fontsize=9)
        ax.set_title(ylabel, fontsize=10)

    # ═══ Common formatting: shared x-limits, outlier vertical lines, grid ══
    for ax in axes:
        ax.set_xlim(xlim_lo, xlim_hi)
        ax.set_xlabel("expMidptMJD", fontsize=8)
        ax.tick_params(axis="both", labelsize=7)
        ax.grid(True, lw=0.3, alpha=0.35)
        for x_out in outlier_mjds_all:
            ax.axvline(x_out, color="red", lw=0.8, ls=":", alpha=0.5, zorder=0)

    fig.tight_layout()
    savefig(fig, file_stem)

    gc.collect()

## 7. Discover per-star files and run the plotting loop

In [ ]:
lc_files = sorted(f for f in os.listdir(DIR_PER_STAR_IN) if f.endswith(PER_STAR_SUFFIX))
log.info("Found %d per-star FGCM-enriched LC files.", len(lc_files))
lc_files

In [ ]:
n_ok = 0
n_err = 0

for fname in lc_files:
    src_path = os.path.join(DIR_PER_STAR_IN, fname)

    try:
        df_star = pd.read_csv(src_path)
    except Exception as exc:
        log.error("  ERROR reading %s: %s", fname, exc)
        n_err += 1
        continue

    simbad_id = (
        df_star["simbad_id"].iloc[0]
        if "simbad_id" in df_star.columns and len(df_star) > 0
        else fname.replace(PER_STAR_SUFFIX, "")
    )

    file_stem = fname.replace(PER_STAR_SUFFIX, "") + "_LC_ug_vs_CALIB_AUXTEL"

    log.info("Plotting: %s  (%d rows)", simbad_id, len(df_star))

    try:
        plot_star_ug_calib_auxtel(df_star, simbad_id, file_stem, df_auxtel)
        n_ok += 1
    except Exception as exc:
        log.error("  ERROR plotting %s: %s", simbad_id, exc)
        plt.close("all")
        n_err += 1

log.info("Done — %d figures saved, %d errors.", n_ok, n_err)

## 8. Quick inline preview of one figure

In [ ]:
# Display the first saved PNG inline for a quick check
from IPython.display import Image, display

saved_pngs = sorted(os.path.join(DIR_FIGS, f) for f in os.listdir(DIR_FIGS) if f.endswith(".png"))

if saved_pngs:
    log.info("Previewing: %s", saved_pngs[0])
    display(Image(filename=saved_pngs[0], width=1200))
else:
    log.warning("No PNG figure found in %s", DIR_FIGS)

## 9. Single-star deep dive

Quick access to one star's figure by `simbad_id`, without re-running the
full loop — useful to iterate on a specific outlier case.


In [ ]:
select_starname = "SDSS J100158.92+021246.1"

target_file = None
for fname in lc_files:
    src_path = os.path.join(DIR_PER_STAR_IN, fname)
    df_tmp = pd.read_csv(src_path, nrows=1)
    if "simbad_id" in df_tmp.columns and df_tmp["simbad_id"].iloc[0] == select_starname:
        target_file = fname
        break

if target_file is None:
    log.warning("Star '%s' not found among per-star files.", select_starname)
else:
    df_star_sel = pd.read_csv(os.path.join(DIR_PER_STAR_IN, target_file))
    file_stem_sel = target_file.replace(PER_STAR_SUFFIX, "") + "_LC_ug_vs_CALIB_AUXTEL"
    plot_star_ug_calib_auxtel(df_star_sel, select_starname, file_stem_sel, df_auxtel)
    plt.show()